In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

MYDTYPE = torch.bfloat16
MYDEVICE = 'cuda:0'


class SiLU(nn.SiLU):
    @property
    def output_multiplier(self) -> float:
        return 1.0
    # end
# end


@dataclass
class SimpleMLPConfig:
    dim_hidden: int
    dim_in: int
    dim_out: int
    bias: bool
    activation: nn.Module
# end


class SimpleMLP(nn.Module):
    def __init__(self, config_mlp):
        super().__init__()

        self.dim_hidden = config_mlp.dim_hidden
        self.dim_in = config_mlp.dim_in
        self.dim_out = config_mlp.dim_out
        self.bias = config_mlp.bias

        self.project_gate = nn.Linear(self.dim_in, self.dim_hidden, bias=self.bias, dtype=MYDTYPE)
        self.project_up = nn.Linear(self.dim_in, self.dim_hidden, bias=self.bias, dtype=MYDTYPE)
        self.project_down = nn.Linear(self.dim_hidden, self.dim_out, bias=self.bias, dtype=MYDTYPE)
        self.activation = config_mlp.activation
    # end

    def forward(self, x):
        return self.project_down(self.activation(self.project_gate(x)) * self.project_up(x))
    # end

    def device(self):
        return next(self.parameters()).device
    # end
# end


In [ ]:
def generate_mask_sequence(unmask):
    L, T = unmask.shape[0], unmask.shape[0]
    a = torch.full((L,), -1, dtype=torch.long)
    a[unmask] = torch.arange(T)
    b = a.view(1, -1) - torch.arange(T).view(-1, 1)
    return b
# end


def generate_y(unmask, h=5):
    a = torch.ones(L, dtype=torch.long, device=MYDEVICE)
    a[unmask] = torch.arange(T, device=MYDEVICE)
    b = a.view(1,-1) - torch.arange(T, device=MYDEVICE).view(-1,1)
    mask_current = (b <= h) & (b > 0)
    b[mask_current] = (h+1 - b[mask_current])
    b[~mask_current] = 0
    b = b/b[0].sum()

    # neg_inf = torch.finfo(b.dtype).min
    # b[~mask_current] = neg_inf
    return b
# end

def soft_rank_loss(pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    logits: [B, L] raw scores from the model
    target: [B, L] soft labels after normalization, each row sums to 1

    returns:
        scalar loss
    """
    log_probs = F.log_softmax(pred, dim=-1)   # [B, L]
    loss = -(y * log_probs).sum(dim=-1)    # [B]
    return loss.mean()


In [2]:
torch.manual_seed(0)

config_mlp1 = SimpleMLPConfig(
    dim_in=3,
    dim_hidden=32,
    dim_out=32,
    bias=True,
    activation=SiLU()
)

config_mlp2 = SimpleMLPConfig(
    dim_in=32,
    dim_hidden=32,
    dim_out=1,
    bias=True,
    activation=SiLU()
)

evaluator = nn.Sequential(
    SimpleMLP(config_mlp1),
    SimpleMLP(config_mlp2)
).to('cuda:0')

optimizer = torch.optim.AdamW(evaluator.parameters(), lr=5e-4, weight_decay=1e-4)


In [ ]:
def run_model(model, optimizer):
    T = L = 64
    id_batch = 0
    id_blk = 0
    folder_base = f'../stats_inf/{id_batch}'
    pos_root = 32
    h=5


    pos_base = pos_root + id_blk * L
    margin = torch.load(f'{folder_base}/margin_{pos_base}_{pos_base + L}.pt').to(dtype=MYDTYPE)
    conf = torch.load(f'{folder_base}/conf_{pos_base}_{pos_base + L}.pt').to(dtype=MYDTYPE)
    attn_last = torch.load(f'{folder_base}/attn_{pos_base}_{pos_base + L}.pt').squeeze(-2)[:,-1,:].to(dtype=MYDTYPE)
    unmask = torch.load(f'{folder_base}/unmask_{pos_base}_{pos_base + L}.pt').squeeze(-1) - pos_base

    x = torch.stack([attn_last, conf, margin], dim=-1)

    y = generate_y(unmask, h)
    output = evaluator(x).squeeze(-1)
    mask_unmask = generate_mask_sequence(unmask) > 0
    losses = []
    for r in range(mask_unmask.shape[0]-h):
        loss = soft_rank_loss(output[r,mask_unmask[r]], y[r,mask_unmask[r]])
        losses.append(loss)
    # end

    losses = torch.stack(losses).mean()
    losses.backward()
    optimizer.step()    